# 02F — Dynamic Recommendation Engine (Final Phase 2)

**Builds on:** `01_core_scoring_engine.ipynb` (Phase 1), `02_dynamic_recommendation_layer.ipynb` (Phase 2 Draft)

**Architecture:** Two-Stage NLP Pipeline — TF-IDF Cosine Filter → KNN Profile Re-ranker

---

### Pipeline Overview

```
User Parameters: anchor_region, group_size, tourist_type
        │
        ▼
   ┌────────────────────────────────────────┐
   │  STAGE 1: TF-IDF + Cosine Similarity   │
   │  Tags → TfidfVectorizer → cosine_sim   │
   │  Hard filter: score == 0 → eliminated  │
   └───────────────┬────────────────────────┘
                   │ survivors only
                   ▼
   ┌────────────────────────────────────────┐
   │  STAGE 2: KNN Profile Re-ranker        │
   │  Numerical features → NearestNeighbors │
   │  Distance → knn_profile_score (0-100)  │
   └───────────────┬────────────────────────┘
                   │
                   ▼
   base_score = (tfidf×0.50) + (knn×0.30) + (attractiveness×0.20)
   final_match_score = base_score - (congestion×0.5) - penalty
```


In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler

import ipywidgets as widgets
from IPython.display import display, clear_output

print("All imports loaded successfully.")

All imports loaded successfully.


---
## Step 1 — 30-Location Japan Tourism Dataset (Expanded Tags)

Each location carries **10–15 detailed tags** spanning three conceptual tiers:

| Tier | Purpose | Examples |
|---|---|---|
| **Core Motivators** | Primary draw | `temple`, `hot_spring`, `urban` |
| **Vibe** | Atmospheric character | `serene`, `lively`, `rustic` |
| **Micro-features** | Specific activities/assets | `matcha`, `sake_brewery`, `cable_car` |

Tags are stored as **lists** for `TfidfVectorizer` compatibility.

In [2]:
# ── 30-Location Japan Tourism Dataset (10-15 Tags Per Location) ──────────────
data = [
    # ═══════════════════════  MAJOR ANCHORS  ═══════════════════════
    {"region": "Tokyo", "is_anchor": True, "tourist_count": 200000, "capacity": 150000,
     "avg_delay_minutes": 25, "cultural_score": 85, "experiential_score": 95,
     "infrastructure_score": 95, "promotion_score": 100, "cleanliness_score": 80,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "tags": ["urban", "modern", "shopping", "food", "nightlife", "anime",
             "tech", "skyscraper", "street_food", "museum", "lively",
             "fashion", "train_hub", "neon", "pop_culture"]},

    {"region": "Kyoto", "is_anchor": True, "tourist_count": 120000, "capacity": 80000,
     "avg_delay_minutes": 30, "cultural_score": 95, "experiential_score": 90,
     "infrastructure_score": 85, "promotion_score": 95, "cleanliness_score": 72,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "tags": ["historic", "temple", "shrine", "cherry_blossom", "traditional",
             "unesco", "culture", "geisha", "gardens", "bamboo",
             "matcha", "photo_spot", "architecture", "autumn_leaves", "crafts"]},

    {"region": "Osaka", "is_anchor": True, "tourist_count": 150000, "capacity": 120000,
     "avg_delay_minutes": 20, "cultural_score": 80, "experiential_score": 95,
     "infrastructure_score": 90, "promotion_score": 90, "cleanliness_score": 75,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "tags": ["urban", "food", "nightlife", "modern", "street_food",
             "castle", "lively", "comedy", "shopping", "takoyaki",
             "neon", "canal", "entertainment"]},

    {"region": "Hakone", "is_anchor": True, "tourist_count": 60000, "capacity": 40000,
     "avg_delay_minutes": 35, "cultural_score": 85, "experiential_score": 90,
     "infrastructure_score": 80, "promotion_score": 85, "cleanliness_score": 78,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "tags": ["hot_spring", "nature", "mountain", "view", "lake",
             "ryokan", "cable_car", "volcanic", "serene", "art_museum",
             "scenic_train", "relaxation", "fuji_view"]},

    {"region": "Nara", "is_anchor": True, "tourist_count": 70000, "capacity": 50000,
     "avg_delay_minutes": 25, "cultural_score": 95, "experiential_score": 85,
     "infrastructure_score": 80, "promotion_score": 80, "cleanliness_score": 74,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": True,
     "tags": ["historic", "temple", "nature", "deer", "unesco",
             "traditional", "park", "buddha", "ancient", "serene",
             "photo_spot", "culture", "architecture"]},

    {"region": "Hiroshima", "is_anchor": True, "tourist_count": 90000, "capacity": 70000,
     "avg_delay_minutes": 20, "cultural_score": 88, "experiential_score": 80,
     "infrastructure_score": 75, "promotion_score": 85, "cleanliness_score": 78,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "tags": ["historic", "memorial", "peace", "temple", "island",
             "unesco", "culture", "coastal", "tram", "okonomiyaki",
             "architecture", "museum", "solemn"]},

    {"region": "Kamakura", "is_anchor": True, "tourist_count": 55000, "capacity": 40000,
     "avg_delay_minutes": 28, "cultural_score": 90, "experiential_score": 85,
     "infrastructure_score": 75, "promotion_score": 80, "cleanliness_score": 75,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": True,
     "tags": ["historic", "temple", "coastal", "buddha", "shrine",
             "hiking", "beach", "traditional", "serene", "photo_spot",
             "architecture", "crafts", "day_trip"]},

    {"region": "Nikko", "is_anchor": True, "tourist_count": 45000, "capacity": 35000,
     "avg_delay_minutes": 30, "cultural_score": 92, "experiential_score": 88,
     "infrastructure_score": 70, "promotion_score": 75, "cleanliness_score": 80,
     "has_coach_parking": True, "group_dining_nearby": False, "requires_long_walk": True,
     "tags": ["historic", "shrine", "mountain", "nature", "unesco",
             "waterfall", "autumn_leaves", "ornate", "cedar", "lake",
             "hiking", "traditional", "architecture"]},

    {"region": "Sapporo", "is_anchor": True, "tourist_count": 80000, "capacity": 70000,
     "avg_delay_minutes": 15, "cultural_score": 75, "experiential_score": 90,
     "infrastructure_score": 85, "promotion_score": 85, "cleanliness_score": 85,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "tags": ["urban", "snow", "food", "festival", "beer",
             "ski", "ramen", "park", "modern", "winter_sport",
             "lively", "night_market", "scenic"]},

    {"region": "Fukuoka", "is_anchor": True, "tourist_count": 65000, "capacity": 60000,
     "avg_delay_minutes": 10, "cultural_score": 80, "experiential_score": 90,
     "infrastructure_score": 88, "promotion_score": 80, "cleanliness_score": 82,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "tags": ["urban", "food", "historic", "shopping", "ramen",
             "shrine", "night_market", "canal", "modern", "lively",
             "street_food", "port"]},

    # ═══════════════════════  ORBIT DESTINATIONS  ═══════════════════════
    {"region": "Kanazawa", "is_anchor": False, "tourist_count": 25000, "capacity": 50000,
     "avg_delay_minutes": 8, "cultural_score": 85, "experiential_score": 80,
     "infrastructure_score": 70, "promotion_score": 60, "cleanliness_score": 88,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "tags": ["historic", "cherry_blossom", "garden", "traditional", "crafts",
             "geisha", "samurai", "museum", "culture", "serene",
             "architecture", "matcha", "autumn_leaves"]},

    {"region": "Takayama", "is_anchor": False, "tourist_count": 18000, "capacity": 40000,
     "avg_delay_minutes": 6, "cultural_score": 80, "experiential_score": 75,
     "infrastructure_score": 65, "promotion_score": 55, "cleanliness_score": 90,
     "has_coach_parking": False, "group_dining_nearby": True, "requires_long_walk": True,
     "tags": ["alpine", "mountain", "rural", "historic", "traditional",
             "festival", "sake_brewery", "edo_town", "crafts", "serene",
             "hiking", "wood_carving", "morning_market"]},

    {"region": "Shirakawa-go", "is_anchor": False, "tourist_count": 22000, "capacity": 20000,
     "avg_delay_minutes": 15, "cultural_score": 90, "experiential_score": 85,
     "infrastructure_score": 50, "promotion_score": 70, "cleanliness_score": 80,
     "has_coach_parking": False, "group_dining_nearby": False, "requires_long_walk": True,
     "tags": ["rural", "historic", "snow", "mountain", "unesco",
             "traditional", "thatched_roof", "village", "photo_spot", "serene",
             "farming", "culture", "winter"]},

    {"region": "Koyasan", "is_anchor": False, "tourist_count": 12000, "capacity": 25000,
     "avg_delay_minutes": 5, "cultural_score": 95, "experiential_score": 85,
     "infrastructure_score": 60, "promotion_score": 50, "cleanliness_score": 85,
     "has_coach_parking": True, "group_dining_nearby": False, "requires_long_walk": True,
     "tags": ["temple", "historic", "mountain", "rural", "unesco",
             "buddhist", "cemetery", "serene", "meditation", "cable_car",
             "traditional", "spiritual", "pilgrim", "shojin_ryori"]},

    {"region": "Nagasaki", "is_anchor": False, "tourist_count": 20000, "capacity": 45000,
     "avg_delay_minutes": 8, "cultural_score": 88, "experiential_score": 85,
     "infrastructure_score": 75, "promotion_score": 65, "cleanliness_score": 82,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "tags": ["historic", "memorial", "coastal", "food", "tram",
             "culture", "church", "port", "island", "museum",
             "solemn", "international", "night_view"]},

    {"region": "Tottori", "is_anchor": False, "tourist_count": 12000, "capacity": 35000,
     "avg_delay_minutes": 4, "cultural_score": 70, "experiential_score": 78,
     "infrastructure_score": 60, "promotion_score": 40, "cleanliness_score": 85,
     "has_coach_parking": True, "group_dining_nearby": False, "requires_long_walk": False,
     "tags": ["rural", "nature", "sand_dunes", "coastal", "adventure",
             "camel_ride", "paragliding", "museum", "quiet", "scenic",
             "seafood", "off_beaten_path"]},

    {"region": "Akita", "is_anchor": False, "tourist_count": 9000, "capacity": 30000,
     "avg_delay_minutes": 3, "cultural_score": 65, "experiential_score": 72,
     "infrastructure_score": 55, "promotion_score": 35, "cleanliness_score": 82,
     "has_coach_parking": False, "group_dining_nearby": False, "requires_long_walk": True,
     "tags": ["rural", "nature", "festival", "mountain", "snow",
             "onsen", "lake", "samurai", "rice_paddy", "quiet",
             "traditional", "rustic"]},

    {"region": "Shimane", "is_anchor": False, "tourist_count": 7000, "capacity": 25000,
     "avg_delay_minutes": 2, "cultural_score": 75, "experiential_score": 70,
     "infrastructure_score": 50, "promotion_score": 30, "cleanliness_score": 87,
     "has_coach_parking": True, "group_dining_nearby": False, "requires_long_walk": False,
     "tags": ["historic", "shrine", "garden", "rural", "mythology",
             "traditional", "serene", "castle", "quiet", "culture",
             "off_beaten_path", "sunset"]},

    {"region": "Okayama", "is_anchor": False, "tourist_count": 15000, "capacity": 40000,
     "avg_delay_minutes": 5, "cultural_score": 82, "experiential_score": 75,
     "infrastructure_score": 70, "promotion_score": 50, "cleanliness_score": 86,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "tags": ["historic", "garden", "castle", "culture", "traditional",
             "crafts", "denim", "fruit", "cycling", "serene",
             "architecture", "museum"]},

    {"region": "Kagawa", "is_anchor": False, "tourist_count": 14000, "capacity": 35000,
     "avg_delay_minutes": 4, "cultural_score": 80, "experiential_score": 82,
     "infrastructure_score": 65, "promotion_score": 55, "cleanliness_score": 84,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "tags": ["food", "coastal", "shrine", "udon", "island",
             "art", "garden", "pilgrimage", "scenic", "cycling",
             "bridge", "traditional"]},

    {"region": "Naoshima", "is_anchor": False, "tourist_count": 11000, "capacity": 15000,
     "avg_delay_minutes": 12, "cultural_score": 88, "experiential_score": 90,
     "infrastructure_score": 55, "promotion_score": 60, "cleanliness_score": 88,
     "has_coach_parking": False, "group_dining_nearby": False, "requires_long_walk": True,
     "tags": ["art", "coastal", "modern", "rural", "museum",
             "architecture", "island", "sculpture", "gallery", "photo_spot",
             "minimalist", "design", "cycling"]},

    {"region": "Matsuyama", "is_anchor": False, "tourist_count": 18000, "capacity": 40000,
     "avg_delay_minutes": 6, "cultural_score": 85, "experiential_score": 82,
     "infrastructure_score": 72, "promotion_score": 50, "cleanliness_score": 85,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "tags": ["hot_spring", "historic", "castle", "tram", "traditional",
             "literature", "garden", "relaxation", "food", "serene",
             "architecture", "culture"]},

    {"region": "Beppu", "is_anchor": False, "tourist_count": 25000, "capacity": 45000,
     "avg_delay_minutes": 10, "cultural_score": 75, "experiential_score": 88,
     "infrastructure_score": 75, "promotion_score": 65, "cleanliness_score": 80,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "tags": ["hot_spring", "nature", "coastal", "volcanic", "steam",
             "relaxation", "ryokan", "sand_bath", "scenic", "food",
             "hell_tour", "unique"]},

    {"region": "Kurokawa Onsen", "is_anchor": False, "tourist_count": 8000, "capacity": 12000,
     "avg_delay_minutes": 4, "cultural_score": 80, "experiential_score": 95,
     "infrastructure_score": 45, "promotion_score": 40, "cleanliness_score": 92,
     "has_coach_parking": False, "group_dining_nearby": False, "requires_long_walk": True,
     "tags": ["hot_spring", "rural", "mountain", "ryokan", "serene",
             "traditional", "nature", "relaxation", "lantern", "rustic",
             "intimate", "off_beaten_path"]},

    {"region": "Kagoshima", "is_anchor": False, "tourist_count": 16000, "capacity": 38000,
     "avg_delay_minutes": 5, "cultural_score": 78, "experiential_score": 85,
     "infrastructure_score": 70, "promotion_score": 50, "cleanliness_score": 83,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "tags": ["nature", "coastal", "mountain", "hot_spring", "volcanic",
             "scenic", "food", "garden", "historic", "tram",
             "island", "adventure", "subtropical"]},

    {"region": "Sendai", "is_anchor": False, "tourist_count": 30000, "capacity": 65000,
     "avg_delay_minutes": 8, "cultural_score": 82, "experiential_score": 80,
     "infrastructure_score": 85, "promotion_score": 60, "cleanliness_score": 84,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "tags": ["urban", "historic", "food", "festival", "castle",
             "shopping", "beef_tongue", "tanabata", "park", "modern",
             "day_trip", "lively"]},

    {"region": "Aomori", "is_anchor": False, "tourist_count": 10000, "capacity": 35000,
     "avg_delay_minutes": 3, "cultural_score": 75, "experiential_score": 80,
     "infrastructure_score": 60, "promotion_score": 45, "cleanliness_score": 86,
     "has_coach_parking": True, "group_dining_nearby": False, "requires_long_walk": False,
     "tags": ["nature", "snow", "rural", "festival", "nebuta",
             "apple", "hiking", "mountain", "lake", "scenic",
             "autumn_leaves", "quiet", "rustic"]},

    {"region": "Nagano", "is_anchor": False, "tourist_count": 22000, "capacity": 50000,
     "avg_delay_minutes": 7, "cultural_score": 85, "experiential_score": 88,
     "infrastructure_score": 75, "promotion_score": 60, "cleanliness_score": 87,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "tags": ["mountain", "snow", "nature", "temple", "ski",
             "monkey_park", "hiking", "alpine", "traditional", "soba",
             "scenic", "onsen", "winter_sport"]},

    {"region": "Matsumoto", "is_anchor": False, "tourist_count": 15000, "capacity": 35000,
     "avg_delay_minutes": 5, "cultural_score": 88, "experiential_score": 80,
     "infrastructure_score": 70, "promotion_score": 55, "cleanliness_score": 88,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "tags": ["castle", "historic", "alpine", "traditional", "culture",
             "crafts", "soba", "museum", "architecture", "serene",
             "mountain", "wasabi", "art"]},

    {"region": "Toyama", "is_anchor": False, "tourist_count": 13000, "capacity": 40000,
     "avg_delay_minutes": 4, "cultural_score": 75, "experiential_score": 85,
     "infrastructure_score": 68, "promotion_score": 50, "cleanliness_score": 85,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "tags": ["alpine", "nature", "coastal", "snow", "scenic",
             "glass_art", "sushi", "firefly_squid", "mountain", "dam",
             "hiking", "modern", "quiet"]},
]

df = pd.DataFrame(data)

# Quick validation
tag_counts = df["tags"].apply(len)
print(f"Locations: {len(df)}  |  Anchors: {df['is_anchor'].sum()}  |  Orbits: {(~df['is_anchor']).sum()}")
print(f"Tags per location - min: {tag_counts.min()}, max: {tag_counts.max()}, mean: {tag_counts.mean():.1f}")
df[["region", "is_anchor", "cultural_score", "tags"]].head(5)


Locations: 30  |  Anchors: 10  |  Orbits: 20
Tags per location - min: 12, max: 15, mean: 12.8


,region,is_anchor,cultural_score,tags
0,Tokyo,True,85,"[urban, modern, shopping, food, nightlife, ani..."
1,Kyoto,True,95,"[historic, temple, shrine, cherry_blossom, tra..."
2,Osaka,True,80,"[urban, food, nightlife, modern, street_food, ..."
3,Hakone,True,85,"[hot_spring, nature, mountain, view, lake, ryo..."
4,Nara,True,95,"[historic, temple, nature, deer, unesco, tradi..."


---
## Step 2 — Phase 1 Helpers: Congestion Index & Dynamic Penalties

Reproduced from Phase 1 for self-contained execution.

- **Piecewise load score:** Six-segment function mapping `tourist_count / capacity` to a 0–100+ congestion signal.
- **Dominance Rule:** When `load_score >= 100`, delay amplifies congestion rather than averaging.
- **Dynamic feasibility penalty:** Group size gates which operational constraints apply.

In [3]:
# ── Phase 1 Congestion Helpers ────────────────────────────────────────────────
def piecewise_load_score(ratio: float) -> float:
    """Piecewise congestion score with no ceiling above ratio >= 1.2."""
    if ratio < 0.3:
        return ratio * (20 / 0.3)
    elif ratio < 0.7:
        return 20 + (ratio - 0.3) * (30 / 0.4)
    elif ratio < 0.9:
        return 50 + (ratio - 0.7) * (25 / 0.2)
    elif ratio < 1.0:
        return 75 + (ratio - 0.9) * (15 / 0.1)
    elif ratio < 1.2:
        return 90 + (ratio - 1.0) * (10 / 0.2)
    else:
        return 100 + ((ratio - 1.2) * 20)


def compute_congestion_index(load_score: float, delay_score: float) -> float:
    """Dominance Rule: delay amplifies when load_score >= 100."""
    if load_score >= 100:
        return load_score + (delay_score * 0.20)
    return (load_score * 0.70) + (delay_score * 0.30)


def congestion_flag(index: float) -> str:
    if index < 20:   return "Severely undertouristed"
    elif index < 40: return "Undertouristed"
    elif index < 75: return "Balanced"
    elif index < 100: return "Approaching overtourism"
    else:             return "Severely overtouristed"


def compute_dynamic_penalty(row: pd.Series, group_size: int) -> int:
    """Group-size-gated feasibility penalty."""
    if group_size >= 15:
        return (
            (0 if row["has_coach_parking"]  else 20) +
            (0 if row["group_dining_nearby"] else 15) +
            (10 if row["requires_long_walk"] else 0)
        )
    else:
        return (5 if row["requires_long_walk"] else 0)


print("Congestion helpers loaded.")


Congestion helpers loaded.


---
## Step 3 — Two-Stage NLP Pipeline

### Stage 1: TF-IDF + Cosine Similarity (Thematic Hard Filter)

Each location's tag list is joined into a single string and passed through `TfidfVectorizer`.
TF-IDF up-weights **rare, distinctive tags** (`geisha`, `firefly_squid`) and down-weights
**ubiquitous tags** (`historic`, `nature`). Cosine similarity between the anchor's TF-IDF
vector and each orbit's vector produces a 0–100 `tfidf_score`. **Orbits scoring 0.0 are
hard-filtered out** — they share no thematic DNA with the anchor.

### Stage 2: KNN Profile Re-ranker

Surviving orbits are evaluated by `NearestNeighbors` on their **numerical quality profile**
(`cultural_score`, `experiential_score`, `cleanliness_score`, `infrastructure_score`).
Cosine distance is converted to a 0–100 `knn_profile_score`, capturing how closely
an orbit's quality profile mirrors the anchor's.

### Final Score (Additive Weighted Sum Model)

$$\text{base\_score} = (\text{tfidf\_score} \times 0.50) + (\text{knn\_profile\_score} \times 0.30) + (\text{attractiveness} \times 0.20)$$

$$\text{final\_match\_score} = \text{base\_score} - (\text{congestion\_index} \times 0.5) - \text{dynamic\_penalty}$$

In [4]:
# ── Two-Stage NLP Recommendation Engine ──────────────────────────────────────

def _compute_tfidf_scores(df: pd.DataFrame, anchor_name: str) -> dict:
    """Stage 1: TF-IDF cosine similarity (anchor vs all). Returns {region: score}."""
    tag_strings = df["tags"].apply(lambda t: " ".join(t))
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(tag_strings)
    anchor_idx = df.index[df["region"] == anchor_name][0]
    cos_sim = cosine_similarity(tfidf_matrix[anchor_idx], tfidf_matrix).flatten()
    return dict(zip(df["region"], np.round(cos_sim * 100, 2)))


def _compute_knn_profile_scores(df: pd.DataFrame, anchor_name: str,
                                 survivor_regions: list) -> dict:
    """Stage 2: KNN cosine-distance similarity on numerical features.
    Only scores survivors from Stage 1. Returns {region: score}."""
    num_cols = ["cultural_score", "experiential_score",
               "cleanliness_score", "infrastructure_score"]
    scaler = MinMaxScaler()
    num_matrix = scaler.fit_transform(df[num_cols].values)

    knn = NearestNeighbors(n_neighbors=len(df), metric="cosine")
    knn.fit(num_matrix)

    anchor_idx = df.index[df["region"] == anchor_name][0]
    distances, indices = knn.kneighbors(num_matrix[anchor_idx].reshape(1, -1))

    sim_map = {}
    for dist, idx in zip(distances.flatten(), indices.flatten()):
        region = df.iloc[idx]["region"]
        if region in survivor_regions:
            sim_map[region] = round((1 - dist) * 100, 2)
    return sim_map


def generate_recommendations(
    df: pd.DataFrame,
    anchor_region: str,
    group_size: int,
    tourist_type: str,
) -> pd.DataFrame:
    """
    Two-Stage Dynamic Recommendation Engine.

    Parameters
    ----------
    df            : Extended DataFrame with tags (list) and numerical scores.
    anchor_region : Name of the selected anchor destination.
    group_size    : Number of people in the travel group.
    tourist_type  : 'international' or 'native'.

    Returns
    -------
    DataFrame of orbit destinations sorted by final_match_score DESC.
    """
    tourist_type = tourist_type.lower().strip()
    if tourist_type not in ("international", "native"):
        raise ValueError("tourist_type must be 'international' or 'native'")

    anchor_row = df.loc[df["region"] == anchor_region]
    if anchor_row.empty:
        raise ValueError(f"Anchor region '{anchor_region}' not found in dataset.")

    # ── Stage 1: TF-IDF cosine similarity ─────────────────────────────────
    tfidf_scores = _compute_tfidf_scores(df, anchor_region)

    # Hard-filter: keep only orbits with tfidf_score > 0
    orbits = df[df["region"] != anchor_region].copy()
    orbits["tfidf_score"] = orbits["region"].map(tfidf_scores)
    survivors = orbits[orbits["tfidf_score"] > 0.0].copy()
    eliminated = orbits[orbits["tfidf_score"] == 0.0].copy()

    if survivors.empty:
        return pd.DataFrame(columns=["region", "tfidf_score", "knn_profile_score",
                                     "attractiveness", "base_score", "congestion_index",
                                     "congestion_flag", "dynamic_penalty", "final_match_score"])

    # ── Stage 2: KNN profile re-ranking ───────────────────────────────────
    survivor_list = survivors["region"].tolist()
    knn_scores = _compute_knn_profile_scores(df, anchor_region, survivor_list)
    survivors["knn_profile_score"] = survivors["region"].map(knn_scores)

    # ── Dynamic attractiveness weights ────────────────────────────────────
    results = []
    for _, row in survivors.iterrows():
        if tourist_type == "international":
            attractiveness = (
                row["cultural_score"]      * 0.40 +
                row["cleanliness_score"]   * 0.30 +
                row["experiential_score"]  * 0.20 +
                row["infrastructure_score"]* 0.10
            )
        else:  # native
            attractiveness = (
                row["experiential_score"]  * 0.40 +
                row["infrastructure_score"]* 0.30 +
                row["cleanliness_score"]   * 0.20 +
                row["cultural_score"]      * 0.10
            )

        # Normalise attractiveness to 0-100 scale (already is, scores are 0-100)
        attr_norm = attractiveness  # already 0-100 range

        # ── Base score (additive weighted sum) ────────────────────────────
        base_score = (
            row["tfidf_score"]        * 0.50 +
            row["knn_profile_score"]  * 0.30 +
            attr_norm                 * 0.20
        )

        # ── Congestion index ──────────────────────────────────────────────
        load_ratio  = row["tourist_count"] / row["capacity"]
        load_score  = piecewise_load_score(load_ratio)
        delay_score = min((row["avg_delay_minutes"] / 45) * 100, 100)
        c_index     = compute_congestion_index(load_score, delay_score)
        c_flag      = congestion_flag(c_index)

        # ── Dynamic penalty ───────────────────────────────────────────────
        d_penalty = compute_dynamic_penalty(row, group_size)

        # ── Final match score ─────────────────────────────────────────────
        final = round(base_score - (c_index * 0.5) - d_penalty, 2)

        results.append({
            "region":             row["region"],
            "tfidf_score":        row["tfidf_score"],
            "knn_profile_score":  row["knn_profile_score"],
            "attractiveness":     round(attr_norm, 2),
            "base_score":         round(base_score, 2),
            "congestion_index":   round(c_index, 2),
            "congestion_flag":    c_flag,
            "dynamic_penalty":    d_penalty,
            "final_match_score":  final,
        })

    output = (
        pd.DataFrame(results)
        .sort_values("final_match_score", ascending=False)
        .reset_index(drop=True)
    )
    return output


print("Two-Stage NLP Engine loaded successfully.")


Two-Stage NLP Engine loaded successfully.


---
## Step 4 — Scenario Execution: Kyoto (International, Coach Tour)

Quick validation of the pipeline with the same parameters used in Phase 2 draft.


In [5]:
# ── Scenario: Kyoto anchor, coach tour, international ─────────────────────────
scenario = generate_recommendations(
    df           = df,
    anchor_region= "Kyoto",
    group_size   = 35,
    tourist_type = "international",
)

DISPLAY = [
    "region", "tfidf_score", "knn_profile_score", "attractiveness",
    "base_score", "congestion_flag", "dynamic_penalty", "final_match_score"
]

scenario[DISPLAY].style \
    .format({
        "tfidf_score":       "{:.2f}",
        "knn_profile_score": "{:.2f}",
        "attractiveness":    "{:.2f}",
        "base_score":        "{:.2f}",
        "final_match_score": "{:.2f}",
    }) \
    .background_gradient(subset=["final_match_score"], cmap="RdYlGn") \
    .background_gradient(subset=["tfidf_score"],       cmap="Blues") \
    .background_gradient(subset=["knn_profile_score"], cmap="PuBuGn") \
    .map(
        lambda v: "color: crimson; font-weight: bold"
            if v == "Severely overtouristed"
        else ("color: darkorange; font-weight: bold"
            if v == "Approaching overtourism" else ""),
        subset=["congestion_flag"]
    ) \
    .set_caption(
        "Phase 2F — Kyoto | Coach Tour (n=35) | International | Two-Stage Pipeline"
    )


,region,tfidf_score,knn_profile_score,attractiveness,base_score,congestion_flag,dynamic_penalty,final_match_score
0,Kanazawa,61.09,75.12,83.40,69.76,Undertouristed,0,54.84
1,Matsumoto,20.32,76.93,84.60,50.16,Undertouristed,0,38.12
2,Okayama,20.63,71.11,80.60,47.77,Undertouristed,0,37.13
3,Matsuyama,14.78,83.32,83.10,49.01,Undertouristed,0,36.07
4,Nagasaki,5.81,91.62,84.30,47.25,Undertouristed,0,33.79
5,Kagawa,8.35,79.72,80.10,44.11,Undertouristed,0,33.15
6,Nagano,8.19,83.09,85.20,46.06,Undertouristed,0,33.05
7,Kagoshima,2.40,82.92,80.10,42.10,Undertouristed,0,30.25
8,Sendai,2.28,83.08,82.50,42.56,Undertouristed,0,28.66
9,Aomori,7.60,64.13,77.80,38.60,Severely undertouristed,15,15.93


---
## Step 5 — Interactive Dashboard

The ipywidgets panel below exposes all three user parameters:
- **Anchor** dropdown (all 30 locations)
- **Group Size** slider (1–50)
- **Tourist Type** dropdown (shifts attractiveness weights)

The dashboard re-runs the full two-stage pipeline on every parameter change.

In [6]:
# ── Interactive Dashboard ─────────────────────────────────────────────────────
out = widgets.Output()

anchor_dropdown = widgets.Dropdown(
    options=sorted(df["region"].unique().tolist()),
    value="Kyoto",
    description="Anchor:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)

group_slider = widgets.IntSlider(
    value=10, min=1, max=50, step=1,
    description="Group Size:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
)

type_dropdown = widgets.Dropdown(
    options=["international", "native"],
    value="international",
    description="Tourist Type:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)


def render_recommendations(anchor_region, group_size, tourist_type):
    results = generate_recommendations(
        df=df, anchor_region=anchor_region,
        group_size=group_size, tourist_type=tourist_type,
    )

    DISPLAY = [
        "region", "tfidf_score", "knn_profile_score", "attractiveness",
        "base_score", "congestion_flag", "dynamic_penalty", "final_match_score",
    ]

    if results.empty:
        with out:
            clear_output(wait=True)
            print("No thematically relevant orbits found for this anchor.")
        return

    styled = (
        results[DISPLAY].style
        .format({
            "tfidf_score":       "{:.2f}",
            "knn_profile_score": "{:.2f}",
            "attractiveness":    "{:.2f}",
            "base_score":        "{:.2f}",
            "final_match_score": "{:.2f}",
        })
        .background_gradient(subset=["final_match_score"], cmap="RdYlGn")
        .background_gradient(subset=["tfidf_score"],       cmap="Blues")
        .background_gradient(subset=["knn_profile_score"], cmap="PuBuGn")
        .map(
            lambda v: "color: crimson; font-weight: bold"
                if v == "Severely overtouristed"
            else ("color: darkorange; font-weight: bold"
                if v == "Approaching overtourism" else ""),
            subset=["congestion_flag"],
        )
        .set_caption(
            f"Phase 2F Recommendations | Anchor: {anchor_region} | "
            f"Group: {group_size} | Type: {tourist_type.capitalize()}"
        )
    )

    with out:
        clear_output(wait=True)
        display(styled)


def _on_change(_):
    render_recommendations(
        anchor_dropdown.value,
        group_slider.value,
        type_dropdown.value,
    )


anchor_dropdown.observe(_on_change, names="value")
group_slider.observe(_on_change,    names="value")
type_dropdown.observe(_on_change,   names="value")

controls = widgets.VBox([
    widgets.HTML("<h3 style='margin-bottom:8px'>Phase 2F — Two-Stage DSS Dashboard</h3>"),
    widgets.HBox([anchor_dropdown, type_dropdown]),
    group_slider,
    widgets.HTML(
        "<small style='color:grey'>Stage 1 (TF-IDF) hard-filters thematically irrelevant orbits. "
        "Stage 2 (KNN) re-ranks survivors by quality-profile similarity. "
        "Group >= 15 activates coach-tour penalties.</small>"
    ),
])

display(controls, out)

# Trigger initial render
render_recommendations(
    anchor_dropdown.value,
    group_slider.value,
    type_dropdown.value,
)


Output()

---
## Architectural Justification — Why TF-IDF + KNN Supersedes Jaccard/Dice

### The Problem with Set-Theoretic Coefficients

Phase 2 Draft used **Jaccard similarity** to gate thematic relevance:

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

While mathematically sound, Jaccard (and its cousin Dice) suffer from two critical failures when applied to rich tag taxonomies:

---

#### Failure 1: The Linear Tax on Tag-Set Size

Jaccard's denominator is $|A \cup B|$. As tag sets grow from 3–4 tags to 10–15 tags, the union inflates rapidly, **diluting even strong overlaps**. Two destinations sharing 6 out of 15 tags yield $J = 6 / 24 = 0.25$ — the same score as sharing 1 out of 4 tags ($J = 1 / 4 = 0.25$). The coefficient cannot distinguish between a genuinely strong but partial match and a trivial match. Dice partially mitigates this with $2|A \cap B|$ in the numerator, but it still treats every shared tag as equally important, which leads directly to Failure 2.

#### Failure 2: All Tags Are Treated Equally

In a 30-location dataset, `historic` appears in 18 locations and `geisha` in 2. Jaccard/Dice weight both identically. A destination sharing `historic` and `nature` with Kyoto scores the same overlap as one sharing `geisha` and `matcha` — despite the latter pair being *far more thematically distinctive*. This produces **generically correct but practically useless** rankings.

---

### Why TF-IDF Solves Both Failures

TF-IDF's **inverse document frequency** term naturally down-weights tags that appear across many locations:

$$\text{IDF}(t) = \log\frac{N}{\text{df}(t)}$$

where $N$ is the total number of locations and $\text{df}(t)$ is the count of locations containing tag $t$.

- `historic` (df=18): $\text{IDF} = \log(30/18) = 0.22$ — near-zero weight
- `geisha` (df=2): $\text{IDF} = \log(30/2) = 1.18$ — high weight

This means the cosine similarity between Kyoto and Kanazawa (sharing `geisha`, `matcha`, `crafts`, `cherry_blossom`, `autumn_leaves`) scores much higher than between Kyoto and a destination sharing only `historic` and `temple` — because the shared tags are **distinctive to that thematic corridor**, not just commonly appearing.

Cosine similarity also normalises by vector magnitude, making it **independent of tag-set size**. A 12-tag location and a 15-tag location are compared purely on directional alignment in TF-IDF space, not penalised by their difference in cardinality.

---

### Why KNN Adds Value as a Re-Ranker

TF-IDF captures *thematic* similarity but is blind to *quality* similarity. Two destinations may share the same tags but differ dramatically in infrastructure, cleanliness, or experiential richness. A visitor drawn to Kyoto's high cultural score and moderate infrastructure should not be sent to a destination with matching tags but collapsing infrastructure.

KNN on normalised numerical features (`cultural_score`, `experiential_score`, `cleanliness_score`, `infrastructure_score`) evaluates **holistic destination quality**. By operating as a Stage 2 re-ranker — scoring only the Stage 1 survivors — KNN avoids the noise of comparing thematically irrelevant destinations and focuses the quality signal on genuinely viable alternatives.

---

### Summary: Phase 2 Evolution

| Phase | Method | Weakness |
|---|---|---|
| Phase 2 Draft | Jaccard on 3–4 tags | Tag-size dilution, no IDF weighting |
| Phase 2F (Final) | TF-IDF + KNN pipeline | ✅ IDF weights rare tags, KNN evaluates quality |

The two-stage architecture cleanly separates *"Is this destination thematically relevant?"* (TF-IDF) from *"Is this destination qualitatively similar?"* (KNN), producing rankings that are both thematically coherent and practically useful for diversion decisions.